In [ ]:
# Uji Gerak Wan 2.2 TI2V-5B - CAHAYA TERAKHIR (S22 — shot terakhir uji gerak)
# Batch kernel: install deps, decode stills, render 1 clip to /kaggle/working
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'diffusers', 'transformers', 'accelerate', 'imageio', 'imageio-ffmpeg', 'sentencepiece', 'ftfy'], check=False)
print('deps ok')


In [ ]:
import pathlib, urllib.request
S = pathlib.Path('/kaggle/working/stills'); S.mkdir(parents=True, exist_ok=True)
BASE = 'https://raw.githubusercontent.com/Trustmindco/uji-gerak-wan22/main/'
FILES = {
    's22': 's22-gelombang-warna.webp',
}
for k, fn in FILES.items():
    dst = S / f'{k}.webp'
    for attempt in range(3):
        try:
            urllib.request.urlretrieve(BASE + fn, dst)
            break
        except Exception as e:
            print('download retry', k, e)
    print(k, '->', dst.stat().st_size, 'bytes')
from PIL import Image
for k in FILES:
    Image.open(S / f'{k}.webp').convert('RGB').save(S / f'{k}.png')
print('stills ready:', sorted(p.name for p in S.iterdir()))


In [ ]:

import gc, os, time, traceback
import torch
from PIL import Image
from diffusers.utils import export_to_video

NEG = 'no warped face, no face change, no outfit change, no extra fingers, no extra limbs, no text, no watermark, no cartoon look, no flicker'
SHOTS = [
    ('s22', 'High aerial drone shot slowly pushing forward over a vast rolling savanna valley and winding river. A living wave of vivid color — emerald green, gold and warm red — sweeps across the land from foreground to background, dissolving the gray mist wherever it passes, the river sparkling as color returns. Cinematic photorealistic fantasy, gentle film grain, smooth cinematic motion.'),
]

MODEL = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
# v7: v6 rendered s18 fine (DONE, 566s) and finished s22's 20 sampling
# steps, then the kernel died (OOM) during s22's VAE decode/export — the
# SECOND shot in a session keeps dying at the decode peak while the FIRST
# shot always succeeds. v7 therefore renders s22 ALONE in a fresh session.
# v6: v5 WORKED for s06 (704x1280, full first config, face-consistent, clip
# verified) but the commit then died at s18 with Kaggle's OOM banner after
# 1006.1s — memory pile-up between shots (decoded frames of the finished
# shot were never freed; cleanup only ran on failure). v6 renders ONLY the
# two remaining shots (s18, s22) and frees frames + clears the CUDA cache
# before and after every shot.
# v5: v4 got INTO sampling (s06 ran 68s) then failed inside pipe() — exact
# exception line never captured. v5 hardens instead of guessing: fp16 (T4 has
# fp16 tensor cores; bf16 is emulated there), VAE slicing+tiling, and a
# per-shot fallback ladder (1280x720/49f -> 832x480/49f -> 832x480/33f) so one
# bad config cannot kill all three shots.
# v4 fix: v3 loaded the I2V pipeline fine but all shots died with
# NameError: name 'ftfy' is not defined (diffusers Wan _encode_prompt calls
# ftfy.fix_text; ftfy is not a hard diffusers dependency) — 'ftfy' added to
# the pip install list in cell 0.
# v3 fix: v2 preferred WanPipeline (text-to-video, no image= kwarg) and only
# fell back to the I2V class on ImportError — backwards. All 3 shots died with
# TypeError: WanPipeline.__call__() got an unexpected keyword argument 'image'.
try:
    from diffusers import WanImageToVideoPipeline as PipeCls
except ImportError:
    from diffusers import AutoPipelineForImageToVideo as PipeCls
print('pipeline class:', PipeCls.__name__)
pipe = PipeCls.from_pretrained(MODEL, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
for _fn in ('enable_tiling', 'enable_slicing'):
    try:
        getattr(pipe.vae, _fn)()
    except Exception:
        pass

CONFIGS = [(1280, 720, 49), (832, 480, 49), (832, 480, 33)]
for key, prompt in SHOTS:
    t0 = time.time()
    gc.collect(); torch.cuda.empty_cache()
    img = Image.open(f'/kaggle/working/stills/{key}.png').convert('RGB')
    done = False
    for (h, w, nf) in CONFIGS:
        try:
            out = pipe(image=img, prompt=prompt, negative_prompt=NEG,
                       height=h, width=w, num_frames=nf,
                       num_inference_steps=20, guidance_scale=5.0).frames[0]
            path = f'/kaggle/working/uji-{key}.mp4'
            export_to_video(out, path, fps=16)
            del out
            gc.collect(); torch.cuda.empty_cache()
            print(f'DONE {key}: {os.path.getsize(path)} bytes at {w}x{h} {nf}f in {time.time()-t0:.0f}s', flush=True)
            done = True
            break
        except Exception:
            print(f'retry {key}: failed at {w}x{h} {nf}f')
            traceback.print_exc()
            gc.collect(); torch.cuda.empty_cache()
    if not done:
        print(f'FAILED {key} after {time.time()-t0:.0f}s (all configs)')
print('ALL FINISHED')


In [ ]:
import os
for f in sorted(os.listdir('/kaggle/working')):
    if f.endswith('.mp4'):
        print(f, os.path.getsize('/kaggle/working/' + f))
